# 08 — RQ3: Generate External AI Essays via API

Notebook wrapper cho `generate_rq3_essays.py`.

Thiết kế full batch:
- đọc **toàn bộ prompt** từ `rq3_generation_config.json`;
- mỗi model sinh `samples_per_prompt_per_model` bài cho **mỗi prompt**;
- `dry-run` không gọi API;
- `resume` mặc định bỏ qua các job đã `success=True` trong CSV;
- raw response và metadata vẫn được lưu bởi script.

> Với cấu hình mặc định hiện tại: **15 prompts × 5 bài/prompt/model × 3 models = 225 jobs**.


In [ ]:
from pathlib import Path
import json
import os
import sys
import subprocess
import pandas as pd
from IPython.display import display

# Tự xác định project root khi notebook nằm ở project root hoặc project/notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

# Colab fallback cho repo của nhóm
COLAB_ROOT = Path("/content/AI-Generated-Text-Essay-Detection")
if not (PROJECT_ROOT / "rq3_generation_config.json").exists() and COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT

SCRIPT_PATH = PROJECT_ROOT / "generate_rq3_essays.py"
CONFIG_PATH = PROJECT_ROOT / "rq3_generation_config.json"
ENV_PATH = PROJECT_ROOT / ".env"

assert SCRIPT_PATH.exists(), f"Missing script: {SCRIPT_PATH}"
assert CONFIG_PATH.exists(), f"Missing config: {CONFIG_PATH}"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    CONFIG = json.load(f)

# Output path được đọc trực tiếp từ config, không hard-code trong notebook.
def resolve_project_path(value):
    p = Path(value)
    return p if p.is_absolute() else PROJECT_ROOT / p

OUTPUT_CSV = resolve_project_path(CONFIG.get("output", {}).get("csv", "data/rq3/generated_essays.csv"))
RAW_JSONL = resolve_project_path(CONFIG.get("output", {}).get("raw_jsonl", "data/rq3/generated_essays_raw.jsonl"))

print("PROJECT_ROOT :", PROJECT_ROOT)
print("SCRIPT_PATH  :", SCRIPT_PATH)
print("CONFIG_PATH  :", CONFIG_PATH)
print("ENV_PATH     :", ENV_PATH)
print("OUTPUT_CSV   :", OUTPUT_CSV)
print("RAW_JSONL    :", RAW_JSONL)


In [ ]:
# Kiểm tra cấu hình full RQ3 trước khi chạy
PROMPTS = CONFIG.get("prompts", [])
MODELS = [m for m in CONFIG.get("models", []) if m.get("enabled", True)]
SAMPLES_PER_PROMPT_PER_MODEL = int(
    CONFIG.get("samples_per_prompt_per_model", CONFIG.get("samples_per_model", 5))
)
TOTAL_JOBS = len(PROMPTS) * len(MODELS) * SAMPLES_PER_PROMPT_PER_MODEL

assert len(PROMPTS) == 15, f"RQ3 hiện yêu cầu đủ 15 prompt, config đang có {len(PROMPTS)}"
assert len({p["prompt_id"] for p in PROMPTS}) == len(PROMPTS), "Duplicate prompt_id trong config"
assert SAMPLES_PER_PROMPT_PER_MODEL >= 1

print(f"Prompts                    : {len(PROMPTS)}")
print(f"Enabled models             : {len(MODELS)}")
print(f"Essays / prompt / model    : {SAMPLES_PER_PROMPT_PER_MODEL}")
print(f"TOTAL PLANNED JOBS         : {TOTAL_JOBS}")

prompt_df = pd.DataFrame(PROMPTS)[["prompt_id", "prompt_name", "prompt_text"]]
model_df = pd.DataFrame([
    {
        "provider": m.get("provider"),
        "model_family": m.get("model_family"),
        "model": m.get("model"),
        "requests_per_minute": m.get("requests_per_minute", CONFIG.get("default_requests_per_minute")),
    }
    for m in MODELS
])

display(prompt_df)
display(model_df)


In [ ]:
# Cài dependency nếu cần (bỏ comment trên Colab)
# !pip install -q -r {PROJECT_ROOT / "requirements_rq3_api.txt"}


## API key setup

Điền khóa thật vào file `.env` ở thư mục gốc dự án (cùng cấp với `requirements_rq3_api.txt`). File mẫu `.env.rq3.example` không được tự động nạp. Nếu thiếu `dotenv`, chạy `%pip install python-dotenv` trong notebook. Không in giá trị API key.


In [ ]:
from dotenv import dotenv_values

# Inspect the same file used by the subprocess, without caching keys in the kernel.
env_values = dotenv_values(ENV_PATH, encoding="utf-8-sig")
default_keys = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "google": "GEMINI_API_KEY"}
required_keys = sorted({m.get("api_key_env") or default_keys[m["provider"]] for m in MODELS})
for name in required_keys:
    value = (os.environ.get(name, env_values.get(name)) or "").strip()
    ready = bool(value) and not value.lower().startswith("your_")
    print(name, "OK" if ready else f"MISSING - fill in {ENV_PATH}")
del env_values


## 1. Dry-run

Dry-run chỉ lập kế hoạch, **không gọi API và không tốn credit**. Notebook không override số bài; script đọc trực tiếp số bài từ JSON.


In [ ]:
RUN_DRY_RUN = True

if RUN_DRY_RUN:
    dry_cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        "--project-root", str(PROJECT_ROOT),
        "--config", str(CONFIG_PATH),
        "--env-file", str(ENV_PATH),
        "--dry-run",
        "--resume",
    ]
    result = subprocess.run(dry_cmd, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"Dry-run failed with exit code {result.returncode}")
else:
    print("RUN_DRY_RUN=False → bỏ qua dry-run.")


## 2. Chạy API thật với resume

Điền API keys vào `.env`, sau đó đổi `RUN_REAL_API = True`.

`--resume` dùng `job_key = provider + model + prompt_id + replicate_idx`. Vì vậy khi notebook bị ngắt và chạy lại, các job đã thành công sẽ được **SKIP**, còn job lỗi/chưa chạy sẽ tiếp tục.


In [ ]:
RUN_REAL_API = False  # Chỉ đổi thành True sau khi dry-run đúng và .env đã có API keys

if RUN_REAL_API:
    real_cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        "--project-root", str(PROJECT_ROOT),
        "--config", str(CONFIG_PATH),
        "--env-file", str(ENV_PATH),
        "--resume",
    ]
    result = subprocess.run(real_cmd, check=False)
    if result.returncode != 0:
        print(f"Batch kết thúc với exit code {result.returncode}. Có thể chạy lại cell này; resume sẽ skip job đã thành công.")
else:
    print("RUN_REAL_API=False → chưa gọi API thật.")


## 3. Kiểm tra tiến độ / resume

Cell này đọc checkpoint CSV và tính số job đã thành công so với tổng số job dự kiến từ config.


In [ ]:
if OUTPUT_CSV.exists():
    df = pd.read_csv(OUTPUT_CSV)
    success_mask = df["success"].astype(str).str.lower().eq("true")
    success_df = df.loc[success_mask].copy()

    # job_key là khóa resume; dùng nunique để tránh đếm trùng nếu file có lịch sử retry.
    completed = success_df["job_key"].nunique() if "job_key" in success_df.columns else int(success_mask.sum())
    remaining = max(TOTAL_JOBS - completed, 0)

    print(f"CSV rows              : {len(df)}")
    print(f"Successful unique jobs: {completed} / {TOTAL_JOBS}")
    print(f"Remaining jobs         : {remaining}")
    print(f"Progress               : {completed / TOTAL_JOBS * 100:.1f}%")

    if not success_df.empty:
        progress = (
            success_df.groupby(["provider", "model", "prompt_id"], dropna=False)["job_key"]
            .nunique()
            .rename("successful_essays")
            .reset_index()
        )
        progress["target_per_prompt"] = SAMPLES_PER_PROMPT_PER_MODEL
        progress["remaining"] = (progress["target_per_prompt"] - progress["successful_essays"]).clip(lower=0)
        display(progress)

    failed = df.loc[~success_mask]
    if len(failed):
        print(f"Failed/error rows recorded: {len(failed)}")
        cols = [c for c in ["provider", "model", "prompt_id", "replicate_idx", "error_type", "error_message"] if c in failed.columns]
        display(failed[cols].tail(20))
else:
    print("Chưa có output CSV:", OUTPUT_CSV)
    print(f"Khi chạy full batch, mục tiêu là {TOTAL_JOBS} successful jobs.")


### Resume thực tế

- Chạy lại cell **Chạy API thật với resume** nếu Colab/Jupyter bị ngắt.
- Không xóa `generated_essays.csv` nếu muốn tiếp tục.
- Job đã `success=True` sẽ không bị gọi API lại.
- Job lỗi có thể được thử lại ở lần chạy sau.
- Chỉ dùng `--no-resume` khi cố ý muốn chạy lại toàn bộ benchmark.
